# Linear and recurrent sequence models on the funding-aligned observation grid

This execution notebook submits the declared linear-baseline and LSTM regression population to
the shared sequence adapter. The adapter seals the exact folds, excludes windows that cross
missing 8-hour observations, persists each checkpoint, and verifies fitted-state digests before
cached reuse. The linear baseline shares that request contract exactly, so any difference in a
later comparison is the architecture rather than the data handling.

**Learning objectives**

- construct a sequence-model request on an explicit observation cadence;
- inspect lookback, gap policy, eligible keys, and checkpoint identity; and
- verify that cached reuse refers to the same persisted fitted state.

**Book reference:** Chapter 19, recurrent neural networks for time series.

**Prerequisites:** finalized crypto labels, features, and purged walk-forward folds; CUDA for the
canonical run.

In [1]:
import os

import polars as pl

from case_studies.crypto_perps_funding.research_workflow import (
    REGRESSION_LABELS,
    declared_contracts,
    freeze_official_model_population,
    model_request_catalog,
    open_study,
    plan_model_catalog,
    plan_specs,
    run_model_plan,
)

In [2]:
EXECUTION_TIER = "canonical"
SUPERSEDES_POPULATION: str = ""
# The generation of this notebook's own checkpoint population that this run replaces, if any.
# Distinct from SUPERSEDES_POPULATION above, which is the case-wide official model population:
# the two are separate declarations and a refit can move either without moving the other.
SUPERSEDES_MODEL_POPULATION: str = ""
WORKSPACE = os.environ.get("ML4T_OUTPUT_DIR", "")
LABELS = REGRESSION_LABELS
PREVIEW_REDUCTIONS = {}
OVERRIDES = {"device": "cuda"}

## Resolve sequence and checkpoint identities

In [3]:
study = open_study(execution_tier=EXECUTION_TIER, workspace=WORKSPACE or None)
official_population = (
    freeze_official_model_population(study, supersedes=SUPERSEDES_POPULATION or None)
    if EXECUTION_TIER == "canonical"
    else None
)
requests = model_request_catalog("deep_learning", labels=LABELS, config_prefix=("nlinear", "lstm"))
requests

family,label,config_name
str,str,str
"""deep_learning""","""fwd_ret_8h""","""nlinear"""
"""deep_learning""","""fwd_ret_8h""","""lstm_h64"""
"""deep_learning""","""fwd_ret_24h""","""nlinear"""
"""deep_learning""","""fwd_ret_24h""","""lstm_h64"""


In [4]:
plan = plan_model_catalog(
    study,
    requests,
    execution_tier=EXECUTION_TIER,
    overrides=OVERRIDES,
    preview_reductions=PREVIEW_REDUCTIONS,
)
# Sequence eligibility follows from the resolved gap policy and lookback, so read both from the
# frozen specification instead of restating the configuration file here.
resolved_preprocessing = [spec["computation"]["preprocessing"] for spec in plan_specs(plan)]
contracts = declared_contracts(plan).with_columns(
    pl.Series("gap_policy", [step["gap_policy"] for step in resolved_preprocessing]),
    pl.Series("lookback", [step["lookback"] for step in resolved_preprocessing]),
)
contracts.select(
    "label",
    "config_name",
    "gap_policy",
    "lookback",
    "checkpoint_value",
    "eligible_rows",
    "training_hash",
)

label,config_name,gap_policy,lookback,checkpoint_value,eligible_rows,training_hash
str,str,str,i64,i64,i64,str
"""fwd_ret_8h""","""nlinear""","""exclude_windows_crossing_missi…",60,5,31885,"""1ffd9f619d8b"""
"""fwd_ret_8h""","""nlinear""","""exclude_windows_crossing_missi…",60,10,31885,"""1ffd9f619d8b"""
"""fwd_ret_8h""","""nlinear""","""exclude_windows_crossing_missi…",60,15,31885,"""1ffd9f619d8b"""
"""fwd_ret_8h""","""nlinear""","""exclude_windows_crossing_missi…",60,20,31885,"""1ffd9f619d8b"""
"""fwd_ret_8h""","""nlinear""","""exclude_windows_crossing_missi…",60,25,31885,"""1ffd9f619d8b"""
…,…,…,…,…,…,…
"""fwd_ret_24h""","""lstm_h64""","""exclude_windows_crossing_missi…",60,80,31831,"""1c7c6b05c230"""
"""fwd_ret_24h""","""lstm_h64""","""exclude_windows_crossing_missi…",60,85,31831,"""1c7c6b05c230"""
"""fwd_ret_24h""","""lstm_h64""","""exclude_windows_crossing_missi…",60,90,31831,"""1c7c6b05c230"""


The complete case-wide population is recorded before the first fit, so a member that later
fails to train cannot quietly disappear from the population it was declared in. This notebook
produces one slice of it, and that slice must lie inside the declaration.

In [5]:
if official_population is not None:
    outside = set(plan.expected_prediction_hashes) - set(official_population.members)
    if outside:
        raise RuntimeError(
            f"{len(outside)} declared checkpoints lie outside the official model population"
        )

## Execute the declared population

In [6]:
execution = run_model_plan(
    plan,
    supersedes=SUPERSEDES_MODEL_POPULATION or None,
    population_name="crypto-lstm-validation-predictions-v1"
    if EXECUTION_TIER == "canonical"
    else None,
)
catalog = execution.catalog_rows.sort("label", "config_name", "checkpoint_value")
if (
    catalog.height != len(plan.expected_prediction_hashes)
    or catalog.filter(~pl.col("complete")).height
):
    raise RuntimeError("sequence baseline and LSTM checkpoint population is incomplete")
catalog.select(
    "label",
    "config_name",
    "checkpoint_value",
    "training_hash",
    "prediction_hash",
    "complete",
)

label,config_name,checkpoint_value,training_hash,prediction_hash,complete
str,str,i64,str,str,bool
"""fwd_ret_24h""","""lstm_h64""",5,"""1c7c6b05c230""","""fb4ec9da03f9""",true
"""fwd_ret_24h""","""lstm_h64""",10,"""1c7c6b05c230""","""0fc08b1fa399""",true
"""fwd_ret_24h""","""lstm_h64""",15,"""1c7c6b05c230""","""987b2d89b376""",true
"""fwd_ret_24h""","""lstm_h64""",20,"""1c7c6b05c230""","""282997a7da20""",true
"""fwd_ret_24h""","""lstm_h64""",25,"""1c7c6b05c230""","""e75cd8a260d0""",true
…,…,…,…,…,…
"""fwd_ret_8h""","""nlinear""",80,"""1ffd9f619d8b""","""bb317da39cf3""",true
"""fwd_ret_8h""","""nlinear""",85,"""1ffd9f619d8b""","""e925ba8e03ae""",true
"""fwd_ret_8h""","""nlinear""",90,"""1ffd9f619d8b""","""3fceffc3f2f7""",true


## Key takeaways and limitations

- Sequence eligibility depends on the declared cadence, not adjacent row positions.
- Missing observations break a lookback window and reset state according to the resolved policy.
- A fixed lookback limits how much earlier information the model can use.